In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

output_notebook()
hv.extension('bokeh')


Loading BokehJS ...

In [ ]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 

# Load the pickle file
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'
cell_df = pd.read_pickle(pickle_file)
print(f"Unified DataFrame loaded from: {pickle_file}")
print(f"DataFrame shape: {cell_df.shape}")
print(cell_df.info())
cell_df.head()

Unified DataFrame loaded from: /home/barak/Projects/population_analysis/data/unified_cell_trial_data/unified_yasmin_cell_trial_data.pkl
DataFrame shape: (5237600, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5237600 entries, 0 to 5237599
Data columns (total 27 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   cell_ID                 int64  
 1   cell_type               object 
 2   maestro_ID              int64  
 3   problem                 object 
 4   grade                   int64  
 5   filename                object 
 6   trial_name              object 
 7   reaction_time           float64
 8   go_cue                  int64  
 9   stop_cue                float64
 10  trial_failed            bool   
 11  ssd_len                 int64  
 12  ssd_number              float64
 13  type                    object 
 14  first_relevant_saccade  object 
 15  segs_durations          object 
 16  segs_times              object 
 17  trial_length

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,2118,msn,1,NaN,8,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[587.37, 842.6700000000001]",ya230528,a,674,ya230528a
1,2119,msn,2,NaN,8,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[48.3, 101.16999999999999, 928.92, 1997.719999...",ya230528,a,674,ya230528a
2,2121,fsn?,4,NaN,7,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[6.0, 34.4, 49.45, 60.300000000000004, 73.4, 9...",ya230528,a,674,ya230528a
3,2122,msn,5,NaN,9,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[1238.43, 2079.4]",ya230528,a,674,ya230528a
4,2123,msn,6,NaN,8,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[290.55, 1375.8999999999999]",ya230528,a,674,ya230528a


In [3]:
cell_df.columns

Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session'],
      dtype='object')

In [4]:
cell_df = cell_df[cell_df['grade'] >= 8].copy().reset_index(drop=True)
cell_df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,2118,msn,1,NaN,8,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[587.37, 842.6700000000001]",ya230528,a,674,ya230528a
1,2119,msn,2,NaN,8,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[48.3, 101.16999999999999, 928.92, 1997.719999...",ya230528,a,674,ya230528a
2,2122,msn,5,NaN,9,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[1238.43, 2079.4]",ya230528,a,674,ya230528a
3,2123,msn,6,NaN,8,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,"[290.55, 1375.8999999999999]",ya230528,a,674,ya230528a
4,2124,msn,7,NaN,8,ya230528a.0674,STOP_L_SSD2,NaN,1316,1424.0,...,2124,0.0,"[[81, 174], [389, 449]]",None,180,[172.7],ya230528,a,674,ya230528a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4621866,3868,hfdp,65,NaN,8,ya230719a.1185,GO_R,19.0,1283,NaN,...,2360,45.0,"[[58, 146], [278, 341], [1068, 1122], [1302, 1...",None,0,[],ya230719,a,1185,ya230719a
4621867,3869,hfdp,66,NaN,8,ya230719a.1185,GO_R,19.0,1283,NaN,...,2360,45.0,"[[58, 146], [278, 341], [1068, 1122], [1302, 1...",None,0,[],ya230719,a,1185,ya230719a
4621868,3870,hfdp,67,NaN,8,ya230719a.1185,GO_R,19.0,1283,NaN,...,2360,45.0,"[[58, 146], [278, 341], [1068, 1122], [1302, 1...",None,0,[],ya230719,a,1185,ya230719a
4621869,3871,hfdp,68,NaN,8,ya230719a.1185,GO_R,19.0,1283,NaN,...,2360,45.0,"[[58, 146], [278, 341], [1068, 1122], [1302, 1...",None,0,[],ya230719,a,1185,ya230719a


In [5]:
cell_df['session'].nunique()


56

In [6]:
## Cell 3: Trial Type Distribution and Success Rates

# Create summary statistics for plotting
trial_summary = cell_df.groupby(['type', 'trial_failed']).size().reset_index(name='count')
trial_summary['outcome'] = trial_summary['trial_failed'].map({False: 'Success', True: 'Failed'})

# Calculate success rates by trial type
success_rates = cell_df.groupby('type').agg({
    'trial_failed': ['count', 'sum', 'mean']
}).round(3)
success_rates.columns = ['total_trials', 'failed_trials', 'failure_rate']
success_rates['success_rate'] = (1 - success_rates['failure_rate']) * 100
success_rates['failure_rate'] *= 100
print("Success rates by trial type:")
print(success_rates)

# Create the main visualization
plot1 = trial_summary.hvplot.bar(
    x='type', y='count', by='outcome',
    stacked=True,
    title=f'{monkey.title()} - Trial Distribution by Type and Outcome',
    xlabel='Trial Type',
    ylabel='Number of Trials',
    width=600, height=400,
    color=['#2E8B57', '#CD5C5C'],  # Green for success, red for failed
    legend='top_right'
)

plot1.opts(
    fontsize={'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12},
)
# success_rates

Success rates by trial type:
      total_trials  failed_trials  failure_rate  success_rate
type                                                         
CONT       1067284         135297          12.7          87.3
GO         2592655         204697           7.9          92.1
STOP        961932         474631          49.3          50.7


:Bars   [type,outcome]   (count)

In [7]:
## Cell 4: Success Rates by Trial Type (Percentage View)
# Create percentage view of success rates
trial_pct = cell_df.groupby('type').apply(
    lambda x: pd.Series({
        'Success': (1 - x['trial_failed'].mean()) * 100,
        'Failed': x['trial_failed'].mean() * 100
    })
).reset_index()

trial_pct_melted = trial_pct.melt(id_vars='type', var_name='outcome', value_name='percentage')

plot2 = trial_pct_melted.hvplot.bar(
    x='type', y='percentage', by='outcome',
    stacked=True,
    title=f'{monkey.title()} - Success Rate by Trial Type (%)',
    xlabel='Trial Type',
    ylabel='Percentage of Trials',
    width=600, height=400,
    color=['#2E8B57', '#CD5C5C'],
    legend='top',
    ylim=(0, 100)
)

plot2
# trial_pct
# trial_pct_melted

/tmp/ipykernel_10411/925790005.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trial_pct = cell_df.groupby('type').apply(


:Bars   [type,outcome]   (percentage)

In [12]:
from bokeh.palettes import Colorblind

# Histogram of cells per grade
print("=== CELLS PER GRADE ANALYSIS ===")

# Get the grade distribution for all cells
grade_counts = cell_df['grade'].value_counts().sort_index()
print(f"Grade distribution:")
for grade, count in grade_counts.items():
    print(f"  Grade {grade}: {count:,} cells")

print(f"\nTotal cells: {len(cell_df):,}")
print(f"Grade range: {cell_df['grade'].min()} - {cell_df['grade'].max()}")
print(f"Mean grade: {cell_df['grade'].mean():.2f}")
print(f"Median grade: {cell_df['grade'].median():.1f}")

# Create bar plot using hvplot with different colors per bar
grade_counts_df = cell_df.groupby('grade').size().reset_index(name='count')

# Create individual bars with different colors
bars = []
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
n_bars = len(grade_counts_df)
palette_size = min(8, max(3, n_bars))
colors = Colorblind[palette_size]

for i, (grade, count) in enumerate(zip(grade_counts_df['grade'], grade_counts_df['count'])):
    bar = hv.Bars([(grade, count)], kdims='grade', vdims='count').opts(
        color=colors[i], 
        alpha=0.8,
        width=10,
    )
    bars.append(bar)

# Overlay all bars
grade_bar = hv.Overlay(bars).opts(
    title=f'{monkey.title()} - Distribution of Cells per Grade',
    xlabel='Grade',
    ylabel='Number of Cells',
    width=700, height=400,
    fontsize=font_dict,
    xticks=list(grade_counts_df['grade'])
)

grade_bar

=== CELLS PER GRADE ANALYSIS ===
Grade distribution:
  Grade 8: 3,057,960 cells
  Grade 9: 1,336,905 cells
  Grade 10: 226,394 cells
  Grade 11: 612 cells

Total cells: 4,621,871
Grade range: 8 - 11
Mean grade: 8.39
Median grade: 8.0


:Overlay
   .Bars.I   :Bars   [grade]   (count)
   .Bars.II  :Bars   [grade]   (count)
   .Bars.III :Bars   [grade]   (count)
   .Bars.IV  :Bars   [grade]   (count)

In [13]:
# Violin plot of cell distribution by trial type and outcome
print("=== CELL DISTRIBUTION BY TRIAL TYPE AND OUTCOME ===")

# Count cells per trial for all trials - using correct column names
cells_per_trial = cell_df.groupby(['filename', 'type', 'trial_failed']).size().reset_index(name='cell_count')

# # Add outcome labels
cells_per_trial['outcome'] = cells_per_trial['trial_failed'].map({False: 'Success', True: 'Failed'})
cells_per_trial
print(f"Total trials analyzed: {len(cells_per_trial):,}")

print(f"\nCells per trial statistics by type and outcome:")
for trial_type in cells_per_trial['type'].unique():
    for outcome in ['Success', 'Failed']:
        type_outcome_data = cells_per_trial[
            (cells_per_trial['type'] == trial_type) & 
            (cells_per_trial['outcome'] == outcome)
        ]['cell_count']
        if len(type_outcome_data) > 0:
            print(f"  {trial_type} {outcome}: Mean={type_outcome_data.mean():.1f}, "
                  f"Median={type_outcome_data.median():.1f}, Min={type_outcome_data.min()}, "
                  f"Max={type_outcome_data.max()}, Trials={len(type_outcome_data)}")

# Create rotated violin plot with split by outcome using hv.Violin
violin_plot = hv.Violin(
    cells_per_trial, kdims=['type', 'outcome'], vdims='cell_count'
).opts(
    opts.Violin(
        show_legend=True, height=500, width=800,
        violin_color=hv.dim('outcome').str(),
        legend_position='top_right',
        split='outcome',
        title=f'{monkey.title()} - Distribution of Cells per Trial by Type and Outcome',
        xlabel='Trial Type',
        ylabel='Number of Cells per Trial',
        show_grid=True,
        violin_width=2,
        invert_axes=True,  # Keep normal orientation (vertical violins)
        tools=['hover'],
        fontsize=font_dict
    )
)


violin_plot

=== CELL DISTRIBUTION BY TRIAL TYPE AND OUTCOME ===
Total trials analyzed: 118,514

Cells per trial statistics by type and outcome:
  GO Success: Mean=39.1, Median=38.0, Min=1, Max=93, Trials=61136
  GO Failed: Mean=38.2, Median=37.0, Min=2, Max=93, Trials=5363
  STOP Success: Mean=37.7, Median=37.0, Min=1, Max=93, Trials=12913
  STOP Failed: Mean=40.4, Median=40.0, Min=2, Max=93, Trials=11745
  CONT Success: Mean=39.1, Median=38.0, Min=1, Max=93, Trials=23834
  CONT Failed: Mean=38.4, Median=37.0, Min=2, Max=93, Trials=3523


:Violin   [type,outcome]   (cell_count)

In [14]:
cell_df['ssd_len'].describe()

count    4.621871e+06
mean     3.138747e+02
std      1.630093e+02
min      2.400000e+01
25%      1.680000e+02
50%      4.500000e+02
75%      4.500000e+02
max      4.500000e+02
Name: ssd_len, dtype: float64

In [15]:
# Bar plot of cell count by cell type and trial type
print("=== CELL TYPE DISTRIBUTION BY TRIAL TYPE ===")

# Get the cell type distribution by trial type
cell_type_trial_counts = cell_df.groupby(['type', 'cell_type']).size().reset_index(name='count')

# Calculate percentages within each trial type
trial_totals = cell_df.groupby('type').size()
cell_type_trial_counts['percentage'] = cell_type_trial_counts.apply(
    lambda row: (row['count'] / trial_totals[row['type']]) * 100, axis=1
)

print(f"Cell type distribution by trial type:")
for trial_type in cell_df['type'].unique():
    print(f"\n{trial_type} trials:")
    trial_data = cell_type_trial_counts[cell_type_trial_counts['type'] == trial_type].sort_values('count', ascending=False)
    for _, row in trial_data.iterrows():
        print(f"  {row['cell_type']}: {row['count']:,} cells ({row['percentage']:.1f}%)")

print(f"\nTotal cells: {len(cell_df):,}")
print(f"Trial types: {cell_df['type'].unique()}")
print(f"Unique cell types: {cell_df['cell_type'].nunique()}")

# Create grouped bar plot with proper separation and legend
# Use hvplot with explicit handling for the legend
cell_type_bar = cell_type_trial_counts.hvplot.bar(
    x='cell_type', y='count', by='type',
    title=f'{monkey.title()} - Distribution of Cells by Cell Type and Trial Type',
    xlabel='Cell Type',
    ylabel='Number of Cells',
    width=1000, height=600,
    alpha=0.8,
    rot=90,
    color=['#2E8B57', '#FF8C00', '#4169E1'],  # Green for GO, Orange for CONT, Blue for STOP
    legend='top_right'
)

# Apply additional styling options
cell_type_bar = cell_type_bar.opts(
    fontsize=font_dict,
    show_legend=True,
    legend_position='top_right',
    legend_opts={'click_policy': 'hide'}
)

cell_type_bar

=== CELL TYPE DISTRIBUTION BY TRIAL TYPE ===
Cell type distribution by trial type:

STOP trials:
  msn: 414,538 cells (43.1%)
  pu msn: 311,180 cells (32.3%)
  hfdp: 85,542 cells (8.9%)
  gpi: 55,466 cells (5.8%)
  pu tan: 31,685 cells (3.3%)
  tan: 31,348 cells (3.3%)
  fb: 19,256 cells (2.0%)
  unknown: 5,054 cells (0.5%)
  fb?: 3,855 cells (0.4%)
  lfd: 2,623 cells (0.3%)
  lfdb: 1,053 cells (0.1%)
  fiber: 193 cells (0.0%)
  fsn: 139 cells (0.0%)

GO trials:
  msn: 1,115,103 cells (43.0%)
  pu msn: 839,875 cells (32.4%)
  hfdp: 230,265 cells (8.9%)
  gpi: 150,519 cells (5.8%)
  pu tan: 85,688 cells (3.3%)
  tan: 84,352 cells (3.3%)
  fb: 52,088 cells (2.0%)
  unknown: 13,472 cells (0.5%)
  fb?: 10,655 cells (0.4%)
  lfd: 6,887 cells (0.3%)
  lfdb: 2,896 cells (0.1%)
  fiber: 537 cells (0.0%)
  fsn: 318 cells (0.0%)

CONT trials:
  msn: 460,390 cells (43.1%)
  pu msn: 344,875 cells (32.3%)
  hfdp: 94,707 cells (8.9%)
  gpi: 61,941 cells (5.8%)
  pu tan: 35,135 cells (3.3%)
  tan: 34

:Bars   [cell_type,type]   (count)

In [16]:
# How many succesful trials per cell type and trial type
print("=== SUCCESSFUL TRIALS PER CELL TYPE AND TRIAL TYPE ===")
# Filter for successful trials only
successful_trials = cell_df[cell_df['trial_failed'] == False]
cell_type_trial_counts = successful_trials.groupby(['type', 'cell_type']).size().reset_index(name='count')

cell_type_trial_counts

# Create grouped bar plot with proper separation and legend
# Use hvplot with explicit handling for the legend
successful_cell_type_bar = cell_type_trial_counts.hvplot.bar(
    x='cell_type', y='count', by='type',
    title=f'{monkey.title()} - Successful Trials by Cell Type and Trial Type',
    xlabel='Cell Type',
    ylabel='Number of Successful Trials',
    width=1000, height=600,
    alpha=0.8,
    rot=90,
    color=['#2E8B57', '#FF8C00', '#4169E1'],  # Green for GO, Orange for CONT, Blue for STOP
    legend='top_right'
)
# Apply additional styling options
successful_cell_type_bar = successful_cell_type_bar.opts(
    fontsize=font_dict,
    show_legend=True,
    legend_position='top_right',
    legend_opts={'click_policy': 'hide'}
)   
successful_cell_type_bar


=== SUCCESSFUL TRIALS PER CELL TYPE AND TRIAL TYPE ===


:Bars   [cell_type,type]   (count)

In [17]:
msn_df = cell_df[cell_df['cell_type'].isin(['msn', 'pu msn'])].copy()
msn_df.attrs['minmum_grade'] = 8
msn_df.attrs['description'] = "DataFrame filtered to include only MSN and PU MSN cell types with a minimum grade of 8."
msn_df.attrs['update_stop_trial_failures_by_saccade_amplitude'] = True
msn_df.attrs['monkey'] = monkey

msn_df.to_pickle(save_path / f'msn_{monkey}_cell_trial_data.pkl')


In [24]:
msn_df['screen_rotation'].value_counts()

screen_rotation
0.0      1186519
45.0     1177710
135.0    1121732
Name: count, dtype: int64